# Logistic Regression - Heart Disease

Explores a tuned Logistic Regression classifier on the cleaned splits.

**Method.** Hyperparameters are tuned with stratified 5-fold `GridSearchCV` on the
**training** set; the tuned model is evaluated on the **validation** set. The
**test** set is deliberately left untouched here - it is reserved for the final
cross-model comparison notebook, so the hold-out is spent only once.

Logistic Regression needs scaled continuous features for stable regularization;
scaling is applied to the continuous columns only, inside the pipeline, via
`dataset.build_scaler`.

In [ ]:
import sys
from pathlib import Path

# Make the scripts/ helpers importable from notebooks/.
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay,
)

from dataset import load_splits, get_xy
from train_models import build_pipeline, MODELS, CV

MODEL_NAME = "logreg"

## Load data

In [ ]:
train, val, test = load_splits()
X_train, y_train = get_xy(train)
X_val, y_val = get_xy(val)

print(f"train {X_train.shape}  val {X_val.shape}  (test held back)")
print(f"train disease rate {y_train.mean():.3f}  val {y_val.mean():.3f}")

## Tune hyperparameters (train, 5-fold CV)

The estimator and grid come straight from the `MODELS` registry in
`train_models.py`, so the notebook and the batch script stay in sync.

In [ ]:
cfg = MODELS[MODEL_NAME]
pipe = build_pipeline(cfg["estimator"], cfg["needs_scaling"], list(X_train.columns))
search = GridSearchCV(pipe, cfg["grid"], cv=CV, scoring="roc_auc", n_jobs=-1)
search.fit(X_train, y_train)

best = search.best_estimator_
print("best params:", search.best_params_)
print(f"cv roc-auc: {search.best_score_:.3f}")

## Validation metrics

In [ ]:
proba = best.predict_proba(X_val)[:, 1]
pred = best.predict(X_val)

metrics = {
    "roc_auc": roc_auc_score(y_val, proba),
    "recall": recall_score(y_val, pred),
    "precision": precision_score(y_val, pred),
    "f1": f1_score(y_val, pred),
    "accuracy": accuracy_score(y_val, pred),
}
pd.Series(metrics, name=MODEL_NAME).round(3)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_val, pred), display_labels=["no disease", "disease"]
).plot(cmap="Blues", colorbar=False)
plt.title(f"{MODEL_NAME} - validation confusion matrix")
plt.show()

## Hyperparameter surface

Logistic Regression is controlled by regularization strength (`C`) and the
`l1_ratio` mix. `l1_ratio=0` behaves like L2, `1` like L1, and values between
them use elastic net regularization.

> Note: strong L1 combined with a very small `C` can drive every coefficient
> to zero, collapsing the model to ROC-AUC 0.5. That degenerate cell in the
> heatmap below is expected behaviour, not a bug.

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
surface = cv_results.pivot_table(
    index="param_model__C",
    columns="param_model__l1_ratio",
    values="mean_test_score",
)
surface.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(surface.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(surface.shape[1]), labels=surface.columns)
ax.set_yticks(range(surface.shape[0]), labels=surface.index)
ax.set_xlabel("l1_ratio")
ax.set_ylabel("C")
ax.set_title("LogReg - CV ROC-AUC by regularization")
fig.colorbar(im, ax=ax, label="CV ROC-AUC")

for i in range(surface.shape[0]):
    for j in range(surface.shape[1]):
        ax.text(j, i, f"{surface.iloc[i, j]:.3f}", ha="center", va="center")

plt.show()

## Coefficients

Coefficients are shown after the pipeline transformation. Continuous features
are standardized, while binary and dummy columns remain on their original 0/1
scale, so coefficient magnitudes are most comparable within each feature type.

In [ ]:
feature_names = best[:-1].get_feature_names_out()
feature_names = [name.replace("scale__", "").replace("remainder__", "") for name in feature_names]

coef = pd.Series(best.named_steps["model"].coef_[0], index=feature_names)
coef.sort_values(key=np.abs, ascending=False).round(3)

In [ ]:
from matplotlib.patches import Patch

top_coef = coef.reindex(coef.abs().sort_values(ascending=False).head(12).index).sort_values()
colors = np.where(top_coef > 0, "tab:red", "tab:blue")

top_coef.plot(kind="barh", color=colors, figsize=(7, 5))
plt.axvline(0, color="gray", lw=1)
plt.xlabel("coefficient")
plt.title("LogReg - largest coefficients")
plt.legend(handles=[
    Patch(color="tab:red", label="raises disease odds"),
    Patch(color="tab:blue", label="lowers disease odds"),
])
plt.show()

## ROC curve (validation)

In [ ]:
fpr, tpr, _ = roc_curve(y_val, proba)
plt.plot(fpr, tpr, label=f"{MODEL_NAME} (AUC={metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], ls="--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("LogReg - validation ROC curve")
plt.legend()
plt.show()

## Notes

- Test set intentionally not touched - see the comparison notebook for the
  single final test evaluation of the selected model.
- This notebook is Logistic Regression specific. To explore another model,
  start from the shared template and swap in a model-appropriate diagnostic.